In [16]:
import pandas as pd
import os

# 1. Menentukan Path Absolut File Anda
file_path = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/BMKG_Earthquake_Catalog.csv'
output_dir = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/Regional_Nodes/'

# Membuat direktori output jika belum ada
os.makedirs(output_dir, exist_ok=True)

# 2. Memuat Data
print("Memuat dataset BMKG...")
df = pd.read_csv(file_path)

# 3. Fungsi Reklasifikasi Berdasarkan Batas Koordinat
def assign_rc(row):
    lat, lon = row['Latitude'], row['Longitude']
    
    # Wilayah Utara (Positif)
    if 0.5 <= lat <= 6.0 and 92.0 <= lon <= 109.0: return 'RC_01'
    if 0.5 <= lat <= 6.0 and 108.5 <= lon <= 132.5: return 'RC_10'
    
    # Wilayah Selatan (Negatif, menggunakan nilai absolut)
    abs_lat = abs(lat)
    if lat <= 0: 
        if 1.0 <= abs_lat <= 3.5 and 92.0 <= lon <= 109.0: return 'RC_06'
        if 3.0 <= abs_lat <= 14.0 and 92.0 <= lon <= 109.0: return 'RC_02'
        if 0.0 <= abs_lat <= 14.0 and 108.5 <= lon <= 114.0: return 'RC_07'
        if 0.0 <= abs_lat <= 7.5 and 113.5 <= lon <= 124.0: return 'RC_04'
        if 7.0 <= abs_lat <= 14.0 and 113.5 <= lon <= 122.5: return 'RC_03'
        if 0.0 <= abs_lat <= 7.5 and 123.5 <= lon <= 132.5: return 'RC_09'
        if 7.0 <= abs_lat <= 14.0 and 122.0 <= lon <= 141.0: return 'RC_08'
        if 6.0 <= abs_lat <= 7.5 and 132.0 <= lon <= 141.0: return 'RC_05'
        
    return 'NC'

# 4. Menerapkan Labeling Baru
print("Mereklasifikasi zona regional untuk setiap event...")
df['Assigned_RC'] = df.apply(assign_rc, axis=1)

# 5. Memecah dan Menyimpan ke File Terpisah untuk Tiap Node Klien FL
print(f"Menyimpan partisi dataset ke: {output_dir}")
for rc_code in [f'RC_{str(i).zfill(2)}' for i in range(1, 11)]:
    subset = df[df['Assigned_RC'] == rc_code]
    filename = os.path.join(output_dir, f"Node_{rc_code}_Catalog.csv")
    
    # Menyimpan file hanya jika ada data di regional tersebut
    if not subset.empty:
        subset.to_csv(filename, index=False)
        print(f" -> {filename} berhasil dibuat ({len(subset)} event)")
    else:
        print(f" -> {rc_code} tidak memiliki event, dilewati.")

print("Distribusi dataset lokal untuk Federated Learning selesai.")

Memuat dataset BMKG...
Mereklasifikasi zona regional untuk setiap event...
Menyimpan partisi dataset ke: /Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/Regional_Nodes/
 -> /Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/Regional_Nodes/Node_RC_01_Catalog.csv berhasil dibuat (13739 event)
 -> /Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/Regional_Nodes/Node_RC_02_Catalog.csv berhasil dibuat (25732 event)
 -> /Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/Regional_Nodes/Node_RC_03_Catalog.csv berhasil dibuat (43997 event)
 -> /Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/Regional_Nodes/Node_RC_04_Catalog.csv berhasil dibuat (27313 event)
 -> /Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/Regional_Nodes/Node_RC_05_Catalog.csv berhasil 

In [21]:
import pandas as pd
import os
from datetime import datetime
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.clients.fdsn.header import FDSNNoDataException
from obspy.taup import TauPyModel
from obspy.geodetics import locations2degrees
from tqdm import tqdm  # Tambahkan import tqdm

# 1. Inisialisasi Klien FDSN dan Model Kecepatan
client = Client("IRIS")  
model = TauPyModel(model="iasp91")

# 2. Memuat Inventaris Stasiun Lokal
file_station = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/katalog_radar_dll/INDONESIA_STATION_INVENTORY_STERIL.csv'
df_station = pd.read_csv(file_station)

# 3. Memuat Katalog Regional
file_catalog = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/Regional_Nodes/Node_RC_07_Catalog.csv'
df_catalog = pd.read_csv(file_catalog)

output_dir = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Waveforms/RC_07/'
os.makedirs(output_dir, exist_ok=True)

# Batas radius lokal (~350 km) dalam derajat
MAX_RADIUS_DEG = 3.5 

# Menentukan jumlah baris yang akan dieksekusi (ubah head(50) menjadi keseluruhan dataframe jika sudah siap)
target_df = df_catalog.head(50) 

# 4. Implementasi tqdm pada loop iterasi
for index, row in tqdm(target_df.iterrows(), total=len(target_df), desc="Mengunduh Waveform", unit="event"):
    event_id = row['Event ID']
    
    try:
        waktu_string = f"{row['Date']} {row['Time (UTC)']}"
        waktu_obj = datetime.strptime(waktu_string, "%d-%b-%Y %H:%M:%S")
        origin_time = UTCDateTime(waktu_obj) 
        
        ev_lat = row['Latitude']
        ev_lon = row['Longitude']
        ev_depth = row['Depth (km)']
        
        # Mencari Stasiun Terdekat dari Inventaris Lokal
        nearby_stations = []
        for _, st_row in df_station.iterrows():
            st_lat = st_row['Latitude']
            st_lon = st_row['Longitude']
            
            dist_deg = locations2degrees(ev_lat, ev_lon, st_lat, st_lon)
            
            if dist_deg <= MAX_RADIUS_DEG:
                nearby_stations.append({
                    'network': st_row['Network'],
                    'station': st_row['Station'],
                    'latitude': st_lat,
                    'longitude': st_lon,
                    'distance': dist_deg
                })
        
        if not nearby_stations:
            tqdm.write(f"[{event_id}] Lewati: Tidak ada stasiun di inventaris dalam radius 350km.")
            continue
            
        # Urutkan dari stasiun paling dekat
        nearby_stations = sorted(nearby_stations, key=lambda x: x['distance'])
        
        download_berhasil = False
        
        # Iterasi Percobaan Unduh
        for st_info in nearby_stations:
            net_code = st_info['network']
            sta_code = st_info['station']
            
            arrivals = model.get_travel_times(source_depth_in_km=ev_depth, 
                                              distance_in_degree=st_info['distance'], 
                                              phase_list=["P", "p"])
            if not arrivals:
                continue
                
            p_arrival_time = origin_time + arrivals[0].time
            window_start = p_arrival_time - 5
            window_end = window_start + 60
            
            try:
                st = client.get_waveforms(net_code, sta_code, "*", "BH?,HH?,EH?,SH?", window_start, window_end)
                
                st.detrend("demean")
                st.filter("bandpass", freqmin=1.0, freqmax=45.0)
                st.resample(100.0)
                st.trim(window_start, window_start + 60, pad=True, fill_value=0)
                
                filename = f"{event_id}_{net_code}_{sta_code}.mseed"
                st.write(os.path.join(output_dir, filename), format="MSEED")
                tqdm.write(f"[{event_id}] BERHASIL: {filename} (Jarak: {st_info['distance']:.2f} derajat)")
                
                download_berhasil = True
                break 
                
            except FDSNNoDataException:
                continue 
            except Exception:
                continue
                
        if not download_berhasil:
            tqdm.write(f"[{event_id}] Gagal: Seluruh stasiun terdekat menghasilkan HTTP 204.")
            
    except Exception as e:
        tqdm.write(f"[{event_id}] Gagal Sistem: {e}")

/opt/homebrew/Caskroom/miniforge/base/envs/waveform/lib/python3.10/site-packages/obspy/clients/fdsn/client.py:251: ObsPyDeprecationWarning: IRIS is now EarthScope, please consider changing the FDSN client short URL to 'EARTHSCOPE'.
  warnings.warn(msg, ObsPyDeprecationWarning)
Mengunduh Waveform:   4%|▍         | 2/50 [00:00<00:06,  7.13event/s]

[BMKG-20060802083511-001] Gagal Sistem: 'Latitude'
[BMKG-20060818174026-001] Gagal Sistem: 'Latitude'


Mengunduh Waveform:   8%|▊         | 4/50 [00:00<00:06,  7.24event/s]

[BMKG-20060819115738-001] Gagal Sistem: 'Latitude'
[BMKG-20071113094032-001] Gagal Sistem: 'Latitude'


Mengunduh Waveform:  12%|█▏        | 6/50 [00:00<00:05,  7.55event/s]

...

[BMKG-20090215040849-001] Gagal Sistem: 'Latitude'
[BMKG-20090215060659-001] Gagal Sistem: 'Latitude'


Mengunduh Waveform:  96%|█████████▌| 48/50 [00:06<00:00,  8.19event/s]

[BMKG-20090215120356-001] Gagal Sistem: 'Latitude'
[BMKG-20090216143629-001] Gagal Sistem: 'Latitude'


Mengunduh Waveform: 100%|██████████| 50/50 [00:06<00:00,  7.91event/s]

[BMKG-20090217034424-001] Gagal Sistem: 'Latitude'
[BMKG-20090218024135-001] Gagal Sistem: 'Latitude'
